In [7]:
import os
from pathlib import Path

# Get the current directory of the notebook
notebook_path = Path.cwd()

ROOT = notebook_path.parent 

# Change the Working Directory for the whole process
os.chdir(ROOT)

print(f"Current Working Directory fixed to: {os.getcwd()}")

Current Working Directory fixed to: /srv/homes/onbo10/thesis_main


+ Select the GPU

In [2]:
from src.utils  import *
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device= get_device()

* load general evaluation constants

In [3]:
from src.evaluation.constants import SIGMAS, IOU_THRESHOLDS

* imports

In [4]:

import torch
from torch.utils.data import DataLoader
from src.evaluation.eval_datasets import *
from src.evaluation.evaluation_utils import *
import numpy as np
import yaml
from types import SimpleNamespace
from hrnet_config import cfg , update_config
from ultralytics import YOLO
from mmengine.dataset import DefaultSampler
from mmengine.runner import load_checkpoint
from mmpose.apis import init_model
from mmpose.datasets import build_dataset
from mmengine.dataset import pseudo_collate



+ Paths and data test list for HRNet

In [9]:
img_root_hrnet= 'data/SurgPose/SurgPose_for_HRNet/Extracted_left_right/extracted_frames'
ann_root_hrnet= 'data/SurgPose/SurgPose_for_HRNet/Extracted_left_right/extracted_bboxes_kpts'
split_path='data/SurgPose/SurgPose_for_HRNet/Extracted_left_right/video_split.yaml'
with open(split_path,'r') as f:
    splits=yaml.safe_load(f)
test_vid_list=splits['test']

+ Paths for yolov8-x-pose

In [10]:
img_root_yolo= 'data/SurgPose/SurgPose_for_YOLO/yolo_formated_surgpose_kpts/images/test'
ann_root_yolo= 'data/SurgPose/SurgPose_for_YOLO/yolo_formated_surgpose_kpts/labels/test'

* Define a path to save the results' logs

In [1]:
log_path= "results/Keypoints_detection/inference_results/evaluation_logs.csv"

+ Names of the Valid test instances

In [7]:
# Surgpose dataset has corrupt bboxes instances, these nees do be cleaned out (method from HRNetEvaluationDataset Class)
# In order to evaluate all methods on the exact same instances we need to detect relevent keys
hrnet_dataset = HRNetEvaluationDataset(img_root_hrnet, ann_root_hrnet, mode='cropped',video_list=test_vid_list)
valid_keys = set()

for sample in hrnet_dataset.samples:
    # Create a unique key: 'video_id/left or right'
    img_name = os.path.basename(sample["img_path"])
    video_id = sample["img_path"].split('/')[-2]
    key = f"{video_id}/{img_name}_{sample['obj_id']}"
    valid_keys.add(key)

print(f"Total instances in HRNet evaluation: {len(valid_keys)}")

Total instances in HRNet evaluation: 3073


+ Evaluate HRNet Cropped input (without object detection predecessor)

In [8]:

# Load finetuned HRNet
cfg_file = 'configs/HRNet/w32_256x192_adam_lr1e-3__cropped_out7-finetune.yaml'
model_weights= '/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/HRNet_one_instance/Experiment1/training_chekpoints2026-01-11_18-05-32/model_epoch200.pth'

args = SimpleNamespace(
        cfg=cfg_file,
        opts=[],
        modelDir='',
        logDir='',
        dataDir='',
        prevModelDir=''
    )
update_config(cfg, args)
model= load_pretrained_HRNet(cfg, model_weights, finetuned=True)
device= get_device()
model.to(device)
model.eval()


PoseHighResolutionNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(i

In [9]:
# Load the test dataset and initiate the dataloader
dataset = HRNetEvaluationDataset(img_root_hrnet,ann_root_hrnet, mode='cropped',video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

=> In this dataset class we are skipping instances where the GT bounding boxes are corrupt (H or W are < 20 px or the area < 400 , or GT kpts not in bbox)

In [10]:

num_images, precision, recall, map50, map50_95 = evaluate_HRNET_cropped(model, test_loader,device ,SIGMAS, IOU_THRESHOLDS)
print("\n" + "="*40)
print("HRNET TEST EVALUATION RESULTS")
print("="*40)
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)


HRNET TEST EVALUATION RESULTS
Total Images: 3073
Precision:    0.9997
Recall:       0.9997
mAP@50:       0.9950
mAP@50-95:    0.9401


* Evaluate ViTPose (without object detection predecessor)

In [11]:


config_file = 'configs/Vitpose/vitpose_surg_7kpt.py'
checkpoint_file = 'results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth'

#Initialize the finetuned model
model = init_model(config_file, checkpoint_file, device=device)
model.eval()
dataset = build_dataset(model.cfg.test_dataloader.dataset)
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    collate_fn=pseudo_collate
)


Loads checkpoint by local backend from path: results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


In [12]:
num_images, precision, recall, map50, map50_95 = evaluate_ViTPose_custom(model,dataloader,device,SIGMAS,IOU_THRESHOLDS)
#Write tgis into a display function at some point
print("\n" + "="*40)
print("ViTpose TEST EVALUATION RESULTS")
print("="*40)
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)


ViTpose TEST EVALUATION RESULTS
Total Images: 3073
Precision:    0.9997
Recall:       0.9997
mAP@50:       0.9950
mAP@50-95:    0.9923


* Evaluate YOLOv8x-pose

In [11]:

model_weights='results/Keypoints_detection/training_results/YOLO_trainings/YOLO_Pose_Experiment1/weights/last.pt'
yolo_model = YOLO(model_weights)
dataset = YoloPoseEvaluationDataset(img_dir=img_root_yolo, label_dir=ann_root_yolo)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)


In [13]:

num_images, precision, recall, map50, map50_95, num_valid = evaluate_YOLO(yolo_model, dataloader, device,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root_hrnet)
yolo_results = {
    "num_images": num_images,
    "num_valid": num_valid,
    "precision": precision,
    "recall": recall,
    "map50": map50,
    "map50_95": map50_95
}
log_evaluation_results("YOLOv8x-pose", model_weights, yolo_results, log_path= log_path)


/srv/homes/onbo10/thesis_main/src/evaluation/evaluation_utils.py:171: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  gt_kpts = torch.tensor(gt_kpts_list, device=device, dtype=torch.float32)



YOLOv8x-pose TEST EVALUATION RESULTS 
Total Valid Instances evaluated: 3073
Total Images: 1884
Precision:    0.8273
Recall:       0.9945
mAP@50:       0.8701
mAP@50-95:    0.6614


* HRNET full image 256*192 with data augmentation

In [14]:
# Load finetuned HRNet
cfg_file = 'configs/HRNet/w32_256x192_adam_lr1e-3_out14-finetune.yaml'
model_weights= '/srv/homes/onbo10/thesis_Ons/HRNet_experiments/HRNet_finetuned/Experiment4/training_chekpoints2026-01-09_17-09-50/model_epoch200.pth'

args = SimpleNamespace(
        cfg=cfg_file,
        opts=[],
        modelDir='',
        logDir='',
        dataDir='',
        prevModelDir=''
    )
update_config(cfg, args)
hrnet_model= load_pretrained_HRNet(cfg, model_weights, finetuned=True)
hrnet_model.to(device)
hrnet_model.eval()

PoseHighResolutionNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(i

In [15]:
# Load the test dataset and initiate the dataloader

input_size = (cfg.MODEL.IMAGE_SIZE[1],cfg.MODEL.IMAGE_SIZE[0])
H_h, W_h= cfg.MODEL.HEATMAP_SIZE[0],cfg.MODEL.HEATMAP_SIZE[1]
dataset = HRNetEvaluationDataset(img_root_hrnet,ann_root_hrnet,input_size=input_size,mode='full', video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

In [16]:

num_images, precision, recall, map50, map50_95, num_valid = evaluate_HRNet_full_image(hrnet_model, test_loader, device ,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root_hrnet, w_m=W_h, h_m=H_h)
print("\n" + "="*40)
print("YOLOv8x-pose TEST EVALUATION RESULTS ")
print("="*40)
print(f"Total Valid Instances evaluated: {num_valid}")
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)


YOLOv8x-pose TEST EVALUATION RESULTS 
Total Valid Instances evaluated: 3073
Total Images: 1884
Precision:    0.9281
Recall:       0.9388
mAP@50:       0.9228
mAP@50-95:    0.4053


* HRNET full image 256*192 without data augmentation

In [17]:
# Load finetuned HRNet
cfg_file = 'configs/HRNet/w32_256x192_adam_lr1e-3_out14-finetune.yaml'
model_weights= 'results/Keypoints_detection/training_results/HRNet_trainings/Experiment1/training_checkpoints2026-02-12_10-49-28/model_epoch200.pth'
args = SimpleNamespace(
        cfg=cfg_file,
        opts=[],
        modelDir='',
        logDir='',
        dataDir='',
        prevModelDir=''
    )
update_config(cfg, args)
hrnet_model= load_pretrained_HRNet(cfg,model_weights, finetuned=True)
hrnet_model.to(device)
hrnet_model.eval()

PoseHighResolutionNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(i

In [18]:
# Load the test dataset and initiate the dataloader

input_size = (cfg.MODEL.IMAGE_SIZE[1],cfg.MODEL.IMAGE_SIZE[0])
H_h, W_h= cfg.MODEL.HEATMAP_SIZE[0],cfg.MODEL.HEATMAP_SIZE[1]

dataset = HRNetEvaluationDataset(img_root_hrnet,ann_root_hrnet,input_size=input_size, mode='full', video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

In [19]:
from src.evaluation.evaluation_utils import evaluate_HRNet_full_image

num_images, precision, recall, map50, map50_95, num_valid = evaluate_HRNet_full_image(hrnet_model, test_loader, device ,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root_hrnet, w_m=W_h, h_m=H_h)
print("\n" + "="*40)
print("YOLOv8x-pose TEST EVALUATION RESULTS ")
print("="*40)
print(f"Total Valid Instances evaluated: {num_valid}")
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)


YOLOv8x-pose TEST EVALUATION RESULTS 
Total Valid Instances evaluated: 3073
Total Images: 1884
Precision:    0.9059
Recall:       0.9300
mAP@50:       0.9031
mAP@50-95:    0.3872


* HRNET full image 704*512 with data augmentation

In [20]:
# Load finetuned HRNet
cfg_file = 'configs/HRNet/w32_704x512_adam_lr1e-3_out14-finetune.yaml'
model_weights= 'results/Keypoints_detection/training_results/HRNet_trainings/Experiment2/training_checkpoints2026-02-12_11-06-15/model_epoch200.pth'
args = SimpleNamespace(
        cfg=cfg_file,
        opts=[],
        modelDir='',
        logDir='',
        dataDir='',
        prevModelDir=''
    )
update_config(cfg, args)
hrnet_model= load_pretrained_HRNet(cfg,model_weights, finetuned=True)
device= get_device()
hrnet_model.to(device)
hrnet_model.eval()

PoseHighResolutionNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(i

In [21]:
# Load the test dataset and initiate the dataloader
input_size = (cfg.MODEL.IMAGE_SIZE[1],cfg.MODEL.IMAGE_SIZE[0])
H_h, W_h= cfg.MODEL.HEATMAP_SIZE[0],cfg.MODEL.HEATMAP_SIZE[1]
dataset = HRNetEvaluationDataset(img_root_hrnet,ann_root_hrnet,input_size=input_size,mode='full', video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

In [22]:
num_images, precision, recall, map50, map50_95, num_valid = evaluate_HRNet_full_image(hrnet_model, test_loader, device ,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root_hrnet, w_m=W_h , h_m=H_h)
print("\n" + "="*40)
print("YOLOv8x-pose TEST EVALUATION RESULTS ")
print("="*40)
print(f"Total Valid Instances evaluated: {num_valid}")
print(f"Total Images: {num_images}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"mAP@50:       {map50:.4f}")
print(f"mAP@50-95:    {map50_95:.4f}")
print("="*40)


YOLOv8x-pose TEST EVALUATION RESULTS 
Total Valid Instances evaluated: 3073
Total Images: 1884
Precision:    0.9884
Recall:       0.9997
mAP@50:       0.9895
mAP@50-95:    0.9013
